In [ ]:
import requests
import pandas as pd

print("requests works")
print("pandas works")

In [ ]:
# CELL 1 - League settings

LEAGUE_ID = 61253603

YEARS = list(range(2019, 2027))

print(YEARS)

In [ ]:
# CELL 2 - ESPN authentication

SWID = "{REDACTED-SWID-SEE-DOTENV-EXAMPLE}"
ESPN_S2 = "REDACTED_ESPN_S2_SEE_DOTENV_EXAMPLE"

cookies = {
    "SWID": SWID,
    "espn_s2": ESPN_S2
}

In [ ]:
# CELL 3 - Test ESPN connection

year = 2025

url = (
    f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
    f"seasons/{year}/segments/0/leagues/{LEAGUE_ID}"
)

params = [
    ("view", "mTeam"),
    ("view", "mMatchupScore")
]

response = requests.get(
    url,
    params=params,
    cookies=cookies
)

print(response.status_code)

In [ ]:
# TEST ONE SEASON

year = 2025

url = (
    f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
    f"seasons/{year}/segments/0/leagues/{LEAGUE_ID}"
)

params = [
    ("view", "mTeam"),
    ("view", "mMatchupScore")
]

response = requests.get(
    url,
    params=params,
    cookies=cookies
)

data = response.json()

print("Teams found:", len(data["teams"]))
print("Matchups found:", len(data["schedule"]))

In [ ]:
# TURN 2025 MATCHUPS INTO A TABLE

games_2025 = []

for game in data["schedule"]:

    home = game.get("home")
    away = game.get("away")

    if not home or not away:
        continue

    if "teamId" not in home or "teamId" not in away:
        continue

    home_score = home.get("totalPoints")
    away_score = away.get("totalPoints")

    if home_score is None or away_score is None:
        continue

    games_2025.append({
        "Year": year,
        "Week": game.get("matchupPeriodId"),
        "Home Team": teams.get(home["teamId"]),
        "Home Score": home_score,
        "Away Team": teams.get(away["teamId"]),
        "Away Score": away_score
    })

games_2025 = pd.DataFrame(games_2025)

games_2025.head(20)

In [ ]:
# BUILD 2025 FFBFFL MATCHUP TABLE

year = 2025

url = (
    f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
    f"seasons/{year}/segments/0/leagues/{LEAGUE_ID}"
)

params = [
    ("view", "mTeam"),
    ("view", "mMatchupScore")
]

response = requests.get(
    url,
    params=params,
    cookies=cookies
)

data = response.json()


# -------------------------
# TEAM NAME LOOKUP
# -------------------------

teams = {}

for team in data["teams"]:
    team_name = team.get("name")

    if not team_name:
        location = team.get("location", "")
        nickname = team.get("nickname", "")
        team_name = f"{location} {nickname}".strip()

    teams[team["id"]] = team_name


# -------------------------
# BUILD MATCHUP TABLE
# -------------------------

games_2025 = []

for game in data["schedule"]:

    home = game.get("home")
    away = game.get("away")

    if not home or not away:
        continue

    if "teamId" not in home or "teamId" not in away:
        continue

    home_score = home.get("totalPoints")
    away_score = away.get("totalPoints")

    if home_score is None or away_score is None:
        continue

    games_2025.append({
        "Year": year,
        "Week": game.get("matchupPeriodId"),
        "Home Team": teams.get(home["teamId"]),
        "Home Score": home_score,
        "Away Team": teams.get(away["teamId"]),
        "Away Score": away_score
    })


games_2025 = pd.DataFrame(games_2025)

print("Games loaded:", len(games_2025))

games_2025.head(20)

In [ ]:
# ============================================================
# PULL COMPLETE FFBFFL HISTORY: 2019–2026
# ============================================================

YEARS = range(2019, 2027)

all_team_games = []
all_matchups = []


def get_season(year):

    # Normal ESPN endpoint
    url = (
        f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
        f"seasons/{year}/segments/0/leagues/{LEAGUE_ID}"
    )

    params = [
        ("view", "mTeam"),
        ("view", "mMatchupScore")
    ]

    response = requests.get(
        url,
        params=params,
        cookies=cookies
    )

    # Fallback for historical seasons
    if response.status_code != 200:

        url = (
            f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
            f"leagueHistory/{LEAGUE_ID}"
        )

        params = [
            ("seasonId", year),
            ("view", "mTeam"),
            ("view", "mMatchupScore")
        ]

        response = requests.get(
            url,
            params=params,
            cookies=cookies
        )

    response.raise_for_status()

    season_data = response.json()

    # ESPN sometimes returns historical seasons inside a list
    if isinstance(season_data, list):
        season_data = season_data[0]

    return season_data


for year in YEARS:

    print(f"Loading {year}...")

    try:
        data = get_season(year)

    except Exception as e:
        print(f"   FAILED: {e}")
        continue


    # ----------------------------------------
    # Team names for this season
    # ----------------------------------------

    teams = {}

    for team in data.get("teams", []):

        team_name = team.get("name")

        if not team_name:
            location = team.get("location", "")
            nickname = team.get("nickname", "")
            team_name = f"{location} {nickname}".strip()

        teams[team["id"]] = team_name


    # ----------------------------------------
    # Matchups
    # ----------------------------------------

    season_games = 0

    for game in data.get("schedule", []):

        # Ignore future / unfinished games
        if game.get("winner") == "UNDECIDED":
            continue

        home = game.get("home")
        away = game.get("away")

        if not home or not away:
            continue

        if "teamId" not in home or "teamId" not in away:
            continue

        home_score = home.get("totalPoints")
        away_score = away.get("totalPoints")

        if home_score is None or away_score is None:
            continue

        home_id = home["teamId"]
        away_id = away["teamId"]

        home_team = teams.get(home_id, f"Team {home_id}")
        away_team = teams.get(away_id, f"Team {away_id}")

        week = game.get("matchupPeriodId")

        home_score = float(home_score)
        away_score = float(away_score)

        margin = abs(home_score - away_score)

        # One row per matchup
        all_matchups.append({
            "Year": year,
            "Week": week,
            "Home Team": home_team,
            "Home Score": home_score,
            "Away Team": away_team,
            "Away Score": away_score,
            "Margin": margin,
            "Combined Score": home_score + away_score
        })

        # One row per TEAM performance
        all_team_games.append({
            "Year": year,
            "Week": week,
            "Team": home_team,
            "Score": home_score,
            "Opponent": away_team,
            "Opponent Score": away_score,
            "Point Differential": home_score - away_score
        })

        all_team_games.append({
            "Year": year,
            "Week": week,
            "Team": away_team,
            "Score": away_score,
            "Opponent": home_team,
            "Opponent Score": home_score,
            "Point Differential": away_score - home_score
        })

        season_games += 1


    print(f"   {season_games} completed matchups")


# ----------------------------------------
# Convert everything to pandas tables
# ----------------------------------------

matchups = pd.DataFrame(all_matchups)
games = pd.DataFrame(all_team_games)

print()
print("DONE")
print("Completed matchups:", len(matchups))
print("Individual team performances:", len(games))

In [ ]:
# CHECK WHICH SEASONS ESPN ASSOCIATES WITH THIS LEAGUE ID

test_year = 2025

url = (
    f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
    f"seasons/{test_year}/segments/0/leagues/{LEAGUE_ID}"
)

response = requests.get(
    url,
    params={"view": "mStatus"},
    cookies=cookies
)

print("Status code:", response.status_code)

status_data = response.json()

print("League ID:", status_data.get("id"))
print("Season:", status_data.get("seasonId"))
print("Previous seasons:", status_data.get("status", {}).get("previousSeasons"))

In [ ]:
# SEE EXACTLY WHICH YEARS RESPOND

for year in range(2019, 2027):

    url = (
        f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
        f"seasons/{year}/segments/0/leagues/{LEAGUE_ID}"
    )

    r = requests.get(
        url,
        params={"view": "mStatus"},
        cookies=cookies
    )

    print(year, "->", r.status_code)

In [ ]:
# ============================================================
# FFBFFL MASTER HISTORY
# 2019–2026
# ============================================================

YEARS = range(2019, 2027)

all_games = []
all_matchups = []

for year in YEARS:

    print(f"Loading {year}...")

    url = (
        f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/"
        f"seasons/{year}/segments/0/leagues/{LEAGUE_ID}"
    )

    params = [
        ("view", "mTeam"),
        ("view", "mMatchupScore")
    ]

    response = requests.get(
        url,
        params=params,
        cookies=cookies
    )

    response.raise_for_status()
    data = response.json()

    # ----------------------------------------
    # Team names
    # ----------------------------------------

    teams = {}

    for team in data.get("teams", []):

        team_name = team.get("name")

        if not team_name:
            location = team.get("location", "")
            nickname = team.get("nickname", "")
            team_name = f"{location} {nickname}".strip()

        teams[team["id"]] = team_name

    completed = 0

    # ----------------------------------------
    # Matchups
    # ----------------------------------------

    for game in data.get("schedule", []):

        # Ignore unfinished/future matchups
        if game.get("winner") == "UNDECIDED":
            continue

        home = game.get("home")
        away = game.get("away")

        if not home or not away:
            continue

        if "teamId" not in home or "teamId" not in away:
            continue

        home_score = home.get("totalPoints")
        away_score = away.get("totalPoints")

        if home_score is None or away_score is None:
            continue

        home_score = float(home_score)
        away_score = float(away_score)

        home_team = teams.get(
            home["teamId"],
            f"Team {home['teamId']}"
        )

        away_team = teams.get(
            away["teamId"],
            f"Team {away['teamId']}"
        )

        week = game.get("matchupPeriodId")

        playoff_type = game.get(
            "playoffTierType",
            "NONE"
        )

        # ----------------------------------------
        # One row per matchup
        # ----------------------------------------

        all_matchups.append({
            "Year": year,
            "Week": week,

            "Home Team": home_team,
            "Home Score": home_score,

            "Away Team": away_team,
            "Away Score": away_score,

            "Margin": abs(home_score - away_score),

            "Combined Score":
                home_score + away_score,

            "Playoff Type": playoff_type
        })

        # ----------------------------------------
        # Home team performance
        # ----------------------------------------

        all_games.append({
            "Year": year,
            "Week": week,
            "Team": home_team,
            "Score": home_score,

            "Opponent": away_team,
            "Opponent Score": away_score,

            "Point Differential":
                home_score - away_score,

            "Result":
                "W" if home_score > away_score
                else "L" if home_score < away_score
                else "T",

            "Playoff Type": playoff_type
        })

        # ----------------------------------------
        # Away team performance
        # ----------------------------------------

        all_games.append({
            "Year": year,
            "Week": week,
            "Team": away_team,
            "Score": away_score,

            "Opponent": home_team,
            "Opponent Score": home_score,

            "Point Differential":
                away_score - home_score,

            "Result":
                "W" if away_score > home_score
                else "L" if away_score < home_score
                else "T",

            "Playoff Type": playoff_type
        })

        completed += 1

    print(f"   {completed} completed matchups")


# ============================================================
# CREATE MASTER TABLES
# ============================================================

games = pd.DataFrame(all_games)
matchups = pd.DataFrame(all_matchups)


print("\n==============================")
print("FFBFFL HISTORY LOADED")
print("==============================")

print("Matchups:", len(matchups))
print("Team performances:", len(games))

print("\nYears:")
print(
    matchups.groupby("Year")
    .size()
)

In [ ]:
games.sort_values(
    "Score",
    ascending=False
).head(20)

In [ ]:
games.sort_values(
    "Score"
).head(20)

In [ ]:
# Highest score in a loss

games[
    games["Result"] == "L"
].sort_values(
    "Score",
    ascending=False
).head(20)

In [ ]:
# Lowest score in a win

games[
    games["Result"] == "W"
].sort_values(
    "Score"
).head(20)

In [ ]:
# Biggest blowouts

matchups.sort_values(
    "Margin",
    ascending=False
).head(20)

In [ ]:
# Closest games

matchups.sort_values(
    "Margin"
).head(20)

In [ ]:
# Highest combined-scoring games

matchups.sort_values(
    "Combined Score",
    ascending=False
).head(20)

In [ ]:
# ============================================================
# 2026 FFBFFL DATA
# ============================================================

games_2026 = games[games["Year"] == 2026].copy()
matchups_2026 = matchups[matchups["Year"] == 2026].copy()

print("2026 completed matchups:", len(matchups_2026))
print("2026 team performances:", len(games_2026))
print("Weeks completed:", sorted(games_2026["Week"].unique()))

In [ ]:
# ============================================================
# 2026 TEAM SUMMARY
# ============================================================

team_summary = (
    games_2026
    .groupby("Team")
    .agg(
        Games=("Score", "count"),
        Wins=("Result", lambda x: (x == "W").sum()),
        Losses=("Result", lambda x: (x == "L").sum()),
        Ties=("Result", lambda x: (x == "T").sum()),

        Points_For=("Score", "sum"),
        Points_Against=("Opponent Score", "sum"),

        Avg_Score=("Score", "mean"),
        Avg_Points_Against=("Opponent Score", "mean"),

        High_Score=("Score", "max"),
        Low_Score=("Score", "min"),

        Median_Score=("Score", "median"),
        Score_SD=("Score", "std"),

        Avg_Margin=("Point Differential", "mean")
    )
    .reset_index()
)

team_summary["Point_Diff"] = (
    team_summary["Points_For"]
    - team_summary["Points_Against"]
)

team_summary["Win_Pct"] = (
    team_summary["Wins"]
    + (team_summary["Ties"] * 0.5)
) / team_summary["Games"]

team_summary["Record"] = (
    team_summary["Wins"].astype(str)
    + "-"
    + team_summary["Losses"].astype(str)
)

# Add ties only if there actually are ties
if team_summary["Ties"].sum() > 0:
    team_summary["Record"] = (
        team_summary["Record"]
        + "-"
        + team_summary["Ties"].astype(str)
    )

team_summary = team_summary.sort_values(
    ["Wins", "Points_For"],
    ascending=[False, False]
)

# Round presentation columns
round_cols = [
    "Points_For",
    "Points_Against",
    "Avg_Score",
    "Avg_Points_Against",
    "High_Score",
    "Low_Score",
    "Median_Score",
    "Score_SD",
    "Avg_Margin",
    "Point_Diff"
]

team_summary[round_cols] = team_summary[round_cols].round(2)

team_summary["Win_Pct"] = team_summary["Win_Pct"].round(3)

display(
    team_summary[
        [
            "Team",
            "Record",
            "Points_For",
            "Points_Against",
            "Point_Diff",
            "Avg_Score",
            "Avg_Points_Against",
            "High_Score",
            "Low_Score",
            "Score_SD"
        ]
    ]
)

In [ ]:
# ============================================================
# 2026 LEAGUE SUMMARY
# ============================================================

league_average = games_2026["Score"].mean()
league_median = games_2026["Score"].median()
league_sd = games_2026["Score"].std()

print("2026 FFBFFL LEAGUE SUMMARY")
print("-----------------------------------")
print(f"Average team score:     {league_average:.2f}")
print(f"Median team score:      {league_median:.2f}")
print(f"Scoring SD:             {league_sd:.2f}")
print(f"Highest team score:     {games_2026['Score'].max():.2f}")
print(f"Lowest team score:      {games_2026['Score'].min():.2f}")
print(f"Average game margin:    {matchups_2026['Margin'].mean():.2f}")
print(f"Average combined score: {matchups_2026['Combined Score'].mean():.2f}")

In [ ]:
# ============================================================
# 2026 RECORD BOOK
# ============================================================

highest = games_2026.loc[
    games_2026["Score"].idxmax()
]

lowest = games_2026.loc[
    games_2026["Score"].idxmin()
]

biggest_blowout = matchups_2026.loc[
    matchups_2026["Margin"].idxmax()
]

closest_game = matchups_2026.loc[
    matchups_2026["Margin"].idxmin()
]

highest_combined = matchups_2026.loc[
    matchups_2026["Combined Score"].idxmax()
]

losses = games_2026[
    games_2026["Result"] == "L"
]

wins = games_2026[
    games_2026["Result"] == "W"
]

highest_loss = losses.loc[
    losses["Score"].idxmax()
]

lowest_win = wins.loc[
    wins["Score"].idxmin()
]


print("2026 FFBFFL RECORD BOOK")
print("=" * 65)

print(
    f"\nHighest Score:\n"
    f"{highest['Team']} — {highest['Score']:.2f}\n"
    f"Week {highest['Week']} vs {highest['Opponent']}"
)

print(
    f"\nLowest Score:\n"
    f"{lowest['Team']} — {lowest['Score']:.2f}\n"
    f"Week {lowest['Week']} vs {lowest['Opponent']}"
)

print(
    f"\nHighest Score in a Loss:\n"
    f"{highest_loss['Team']} — {highest_loss['Score']:.2f}\n"
    f"Week {highest_loss['Week']} vs "
    f"{highest_loss['Opponent']} "
    f"({highest_loss['Opponent Score']:.2f})"
)

print(
    f"\nLowest Score in a Win:\n"
    f"{lowest_win['Team']} — {lowest_win['Score']:.2f}\n"
    f"Week {lowest_win['Week']} vs "
    f"{lowest_win['Opponent']} "
    f"({lowest_win['Opponent Score']:.2f})"
)

print(
    f"\nBiggest Blowout:\n"
    f"{biggest_blowout['Home Team']} "
    f"{biggest_blowout['Home Score']:.2f} - "
    f"{biggest_blowout['Away Score']:.2f} "
    f"{biggest_blowout['Away Team']}\n"
    f"Week {biggest_blowout['Week']} | "
    f"Margin: {biggest_blowout['Margin']:.2f}"
)

print(
    f"\nClosest Game:\n"
    f"{closest_game['Home Team']} "
    f"{closest_game['Home Score']:.2f} - "
    f"{closest_game['Away Score']:.2f} "
    f"{closest_game['Away Team']}\n"
    f"Week {closest_game['Week']} | "
    f"Margin: {closest_game['Margin']:.2f}"
)

print(
    f"\nHighest-Scoring Matchup:\n"
    f"{highest_combined['Home Team']} "
    f"{highest_combined['Home Score']:.2f} - "
    f"{highest_combined['Away Score']:.2f} "
    f"{highest_combined['Away Team']}\n"
    f"Week {highest_combined['Week']} | "
    f"Combined: {highest_combined['Combined Score']:.2f}"
)

In [ ]:
# ============================================================
# 2026 ALL-PLAY RECORD
# ============================================================

all_play_rows = []

for week in sorted(games_2026["Week"].unique()):

    week_data = games_2026[
        games_2026["Week"] == week
    ].copy()

    for _, team_row in week_data.iterrows():

        other_scores = week_data[
            week_data["Team"] != team_row["Team"]
        ]["Score"]

        all_play_wins = (
            other_scores < team_row["Score"]
        ).sum()

        all_play_losses = (
            other_scores > team_row["Score"]
        ).sum()

        all_play_ties = (
            other_scores == team_row["Score"]
        ).sum()

        all_play_rows.append({
            "Week": week,
            "Team": team_row["Team"],
            "All_Play_Wins": all_play_wins,
            "All_Play_Losses": all_play_losses,
            "All_Play_Ties": all_play_ties
        })


all_play = pd.DataFrame(all_play_rows)

all_play_summary = (
    all_play
    .groupby("Team")
    .agg(
        All_Play_Wins=("All_Play_Wins", "sum"),
        All_Play_Losses=("All_Play_Losses", "sum"),
        All_Play_Ties=("All_Play_Ties", "sum")
    )
    .reset_index()
)

all_play_summary["All_Play_Win_Pct"] = (
    all_play_summary["All_Play_Wins"]
    + 0.5 * all_play_summary["All_Play_Ties"]
) / (
    all_play_summary["All_Play_Wins"]
    + all_play_summary["All_Play_Losses"]
    + all_play_summary["All_Play_Ties"]
)

all_play_summary = all_play_summary.sort_values(
    "All_Play_Win_Pct",
    ascending=False
)

all_play_summary["All_Play_Win_Pct"] = (
    all_play_summary["All_Play_Win_Pct"]
    .round(3)
)

display(all_play_summary)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["axes.facecolor"] = "#111111"
plt.rcParams["figure.facecolor"] = "#111111"
plt.rcParams["savefig.facecolor"] = "#111111"
plt.rcParams["text.color"] = "white"
plt.rcParams["axes.labelcolor"] = "white"
plt.rcParams["xtick.color"] = "white"
plt.rcParams["ytick.color"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["font.size"] = 11

# 2026 subsets
games_2026 = games[games["Year"] == 2026].copy()
matchups_2026 = matchups[matchups["Year"] == 2026].copy()

In [ ]:
team_summary = (
    games_2026
    .groupby("Team")
    .agg(
        Games=("Score", "count"),
        Wins=("Result", lambda x: (x == "W").sum()),
        Losses=("Result", lambda x: (x == "L").sum()),
        Ties=("Result", lambda x: (x == "T").sum()),
        Points_For=("Score", "sum"),
        Points_Against=("Opponent Score", "sum"),
        Avg_Score=("Score", "mean"),
        High_Score=("Score", "max"),
        Low_Score=("Score", "min"),
        Score_SD=("Score", "std")
    )
    .reset_index()
)

team_summary["Point_Diff"] = team_summary["Points_For"] - team_summary["Points_Against"]
team_summary["Record"] = team_summary["Wins"].astype(str) + "-" + team_summary["Losses"].astype(str)

all_play_summary = (
    all_play
    .groupby("Team")
    .agg(
        All_Play_Wins=("All_Play_Wins", "sum"),
        All_Play_Losses=("All_Play_Losses", "sum"),
        All_Play_Ties=("All_Play_Ties", "sum")
    )
    .reset_index()
)

all_play_summary["All_Play_Win_Pct"] = (
    all_play_summary["All_Play_Wins"] + 0.5 * all_play_summary["All_Play_Ties"]
) / (
    all_play_summary["All_Play_Wins"] +
    all_play_summary["All_Play_Losses"] +
    all_play_summary["All_Play_Ties"]
)

summary = team_summary.merge(all_play_summary, on="Team", how="left")

summary["Expected_Wins"] = summary["All_Play_Win_Pct"] * summary["Games"]
summary["Schedule_Luck"] = summary["Wins"] - summary["Expected_Wins"]

summary = summary.sort_values(["Wins", "Points_For"], ascending=[False, False]).reset_index(drop=True)

num_cols = [
    "Points_For", "Points_Against", "Avg_Score", "High_Score",
    "Low_Score", "Score_SD", "Point_Diff", "All_Play_Win_Pct",
    "Expected_Wins", "Schedule_Luck"
]

summary[num_cols] = summary[num_cols].round(2)

summary

In [ ]:
table_view = summary[
    [
        "Team", "Record", "Points_For", "Points_Against",
        "Point_Diff", "Avg_Score", "High_Score", "Low_Score",
        "Score_SD", "All_Play_Win_Pct", "Expected_Wins", "Schedule_Luck"
    ]
].copy()

styled_table = (
    table_view.style
    .format({
        "Points_For": "{:.2f}",
        "Points_Against": "{:.2f}",
        "Point_Diff": "{:.2f}",
        "Avg_Score": "{:.2f}",
        "High_Score": "{:.2f}",
        "Low_Score": "{:.2f}",
        "Score_SD": "{:.2f}",
        "All_Play_Win_Pct": "{:.3f}",
        "Expected_Wins": "{:.2f}",
        "Schedule_Luck": "{:+.2f}"
    })
    .background_gradient(subset=["Avg_Score"], cmap="Blues")
    .background_gradient(subset=["Point_Diff"], cmap="RdYlGn")
    .background_gradient(subset=["Schedule_Luck"], cmap="RdYlGn")
    .set_properties(**{
        "background-color": "#111111",
        "color": "white",
        "border-color": "#333333",
        "font-size": "11pt"
    })
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#1f1f1f"),
                                     ("color", "white"),
                                     ("border", "1px solid #333333"),
                                     ("font-size", "11pt")]},
        {"selector": "td", "props": [("border", "1px solid #333333")]},
        {"selector": "caption", "props": [("color", "white"),
                                          ("font-size", "14pt"),
                                          ("font-weight", "bold")]}
    ])
    .set_caption("FFBFFL 2026 Summary Through Week 4")
)

display(styled_table)

In [ ]:
plot_df = summary.sort_values("Avg_Score", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(plot_df["Team"], plot_df["Avg_Score"])

league_avg = plot_df["Avg_Score"].mean()
ax.axvline(league_avg, linestyle="--", linewidth=2)

for i, v in enumerate(plot_df["Avg_Score"]):
    ax.text(v + 0.5, i, f"{v:.2f}", va="center", fontsize=10)

ax.set_title("FFBFFL 2026 Scoring Power Rankings", fontsize=18, weight="bold", pad=15)
ax.set_xlabel("Average Points Per Game")
ax.set_ylabel("")
ax.grid(axis="x", linestyle=":", alpha=0.3)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

x = summary["Points_For"]
y = summary["Points_Against"]

ax.scatter(x, y, s=120)

for _, row in summary.iterrows():
    ax.text(row["Points_For"] + 1.5, row["Points_Against"] + 1.5, row["Team"], fontsize=9)

x_avg = summary["Points_For"].mean()
y_avg = summary["Points_Against"].mean()

ax.axvline(x_avg, linestyle="--", linewidth=1.5)
ax.axhline(y_avg, linestyle="--", linewidth=1.5)

ax.set_title("FFBFFL 2026: Points For vs Points Against", fontsize=18, weight="bold", pad=15)
ax.set_xlabel("Points For")
ax.set_ylabel("Points Against")
ax.grid(linestyle=":", alpha=0.3)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
plot_df = summary.sort_values("Schedule_Luck", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(plot_df["Team"], plot_df["Schedule_Luck"])

ax.axvline(0, linewidth=2)

for i, v in enumerate(plot_df["Schedule_Luck"]):
    ax.text(v + (0.03 if v >= 0 else -0.18), i, f"{v:+.2f}", va="center", fontsize=10)

ax.set_title("FFBFFL 2026 Schedule Luck", fontsize=18, weight="bold", pad=15)
ax.set_xlabel("Actual Wins - Expected Wins")
ax.set_ylabel("")
ax.grid(axis="x", linestyle=":", alpha=0.3)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
plot_df = summary.sort_values("Expected_Wins", ascending=True)

y = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(y - 0.2, plot_df["Wins"], height=0.4, label="Actual Wins")
ax.barh(y + 0.2, plot_df["Expected_Wins"], height=0.4, label="Expected Wins")

ax.set_yticks(y)
ax.set_yticklabels(plot_df["Team"])

ax.set_title("FFBFFL 2026 Actual Wins vs Expected Wins", fontsize=18, weight="bold", pad=15)
ax.set_xlabel("Wins")
ax.set_ylabel("")
ax.legend()
ax.grid(axis="x", linestyle=":", alpha=0.3)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
weekly_scores = (
    games_2026.pivot_table(
        index="Week",
        columns="Team",
        values="Score",
        aggfunc="mean"
    )
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 8))

for team in weekly_scores.columns:
    ax.plot(weekly_scores.index, weekly_scores[team], marker="o", linewidth=2, label=team)

ax.set_title("FFBFFL 2026 Weekly Scoring Trends", fontsize=18, weight="bold", pad=15)
ax.set_xlabel("Week")
ax.set_ylabel("Points")
ax.set_xticks(sorted(games_2026["Week"].unique()))
ax.grid(linestyle=":", alpha=0.3)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

week_list = sorted(games_2026["Week"].unique())
data = [games_2026.loc[games_2026["Week"] == w, "Score"] for w in week_list]

ax.boxplot(data, tick_labels=week_list)

ax.set_title("FFBFFL 2026 Weekly Score Distribution", fontsize=18, weight="bold", pad=15)
ax.set_xlabel("Week")
ax.set_ylabel("Team Score")
ax.grid(axis="y", linestyle=":", alpha=0.3)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# Change this if you want another folder
OUTPUT_DIR = r"C:\Users\DCifelli2\Desktop\FFBFFL_Charts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Use your existing summary table
# Must contain:
# Team, Points_For, Points_Against, Wins, Expected_Wins, Schedule_Luck

# If summary already exists, this just makes sure it has what we need
plot_df = summary.copy()

# General plotting style
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

BG = "#0f1117"
AX_BG = "#151922"
GRID = "#3b4252"
TEXT = "#f5f7fa"
MUTED = "#b8c0cc"
BLUE = "#4ea1ff"
LIGHT_BLUE = "#8fc7ff"
GOLD = "#f0c75e"
RED = "#ff6b6b"
GREEN = "#5bd38d"
WHITE = "#ffffff"

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(BG)
ax.set_facecolor(AX_BG)

x = plot_df["Points_For"]
y = plot_df["Points_Against"]

x_avg = x.mean()
y_avg = y.mean()

# Axis padding
x_pad = (x.max() - x.min()) * 0.08
y_pad = (y.max() - y.min()) * 0.08

x_min, x_max = x.min() - x_pad, x.max() + x_pad
y_min, y_max = y.min() - y_pad, y.max() + y_pad

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# Quadrant shading
ax.axvspan(x_avg, x_max, ymin=0, ymax=0.5, alpha=0.10)
ax.axvspan(x_avg, x_max, ymin=0.5, ymax=1, alpha=0.06)
ax.axvspan(x_min, x_avg, ymin=0, ymax=0.5, alpha=0.05)
ax.axvspan(x_min, x_avg, ymin=0.5, ymax=1, alpha=0.12)

# Scatter sizes based on points scored
sizes = 120 + (plot_df["Points_For"] - plot_df["Points_For"].min()) * 2.2

ax.scatter(
    plot_df["Points_For"],
    plot_df["Points_Against"],
    s=sizes,
    alpha=0.92,
    edgecolors=WHITE,
    linewidths=1.2
)

# Team labels
for _, row in plot_df.iterrows():
    ax.text(
        row["Points_For"] + 1.2,
        row["Points_Against"] + 1.2,
        row["Team"],
        color=TEXT,
        fontsize=9,
        ha="left",
        va="bottom",
        path_effects=[pe.withStroke(linewidth=2.5, foreground=BG)]
    )

# Average lines
ax.axvline(x_avg, color=GOLD, linestyle="--", linewidth=1.8)
ax.axhline(y_avg, color=GOLD, linestyle="--", linewidth=1.8)

# Quadrant labels
quad_kw = dict(color=MUTED, fontsize=10, ha="center", va="center")

ax.text((x_avg + x_max)/2, (y_min + y_avg)/2,
        "High Scoring\nEasier Schedule", **quad_kw)

ax.text((x_avg + x_max)/2, (y_avg + y_max)/2,
        "High Scoring\nHarder Schedule", **quad_kw)

ax.text((x_min + x_avg)/2, (y_min + y_avg)/2,
        "Lower Scoring\nEasier Schedule", **quad_kw)

ax.text((x_min + x_avg)/2, (y_avg + y_max)/2,
        "Lower Scoring\nHarder Schedule", **quad_kw)

# Titles
ax.set_title("FFBFFL 2026 — Points For vs Points Against", color=TEXT, pad=16)
ax.text(
    0.5, 1.02,
    "Through Week 4 | Teams in the upper-right are scoring well despite facing stronger weekly opposition",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    color=MUTED,
    fontsize=10
)

# Labels
ax.set_xlabel("Points For", color=TEXT, labelpad=10)
ax.set_ylabel("Points Against", color=TEXT, labelpad=10)

# Grid / spines
ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.45, color=GRID)
for spine in ax.spines.values():
    spine.set_color("#5a6270")
    spine.set_linewidth(1)

ax.tick_params(colors=TEXT)

# Average labels
ax.text(
    x_avg + 1.5,
    y_max - 2,
    f"League Avg PF: {x_avg:.1f}",
    color=GOLD,
    fontsize=9,
    ha="left",
    va="top"
)

ax.text(
    x_max - 1.5,
    y_avg + 1.5,
    f"League Avg PA: {y_avg:.1f}",
    color=GOLD,
    fontsize=9,
    ha="right",
    va="bottom"
)

plt.tight_layout()

scatter_path = os.path.join(OUTPUT_DIR, "ffbffl_2026_points_for_vs_against.png")
plt.savefig(scatter_path, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print("Saved:", scatter_path)

In [ ]:
luck_df = plot_df[["Team", "Wins", "Expected_Wins", "Schedule_Luck"]].copy()
luck_df = luck_df.sort_values("Schedule_Luck", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(BG)
ax.set_facecolor(AX_BG)

# Color by positive / negative luck
bar_colors = [RED if v < 0 else GREEN for v in luck_df["Schedule_Luck"]]

bars = ax.barh(
    luck_df["Team"],
    luck_df["Schedule_Luck"],
    color=bar_colors,
    edgecolor=WHITE,
    linewidth=0.6,
    alpha=0.9
)

# Zero line
ax.axvline(0, color=GOLD, linewidth=2.2)

# Value labels
for bar, (_, row) in zip(bars, luck_df.iterrows()):
    value = row["Schedule_Luck"]
    y = bar.get_y() + bar.get_height() / 2

    if value >= 0:
        ax.text(
            value + 0.03,
            y,
            f"{value:+.2f}",
            va="center",
            ha="left",
            color=TEXT,
            fontsize=10,
            fontweight="bold"
        )
    else:
        ax.text(
            value - 0.03,
            y,
            f"{value:+.2f}",
            va="center",
            ha="right",
            color=TEXT,
            fontsize=10,
            fontweight="bold"
        )

# Secondary annotations: actual vs expected wins
for i, row in luck_df.iterrows():
    ax.text(
        ax.get_xlim()[0] if i == 0 else 0,  # harmless placeholder; replaced after set_xlim below
        i,
        "",
    )

# Better x limits
max_abs = max(abs(luck_df["Schedule_Luck"].min()), abs(luck_df["Schedule_Luck"].max()))
x_lim = max_abs + 0.45
ax.set_xlim(-x_lim, x_lim)

# Now add actual vs expected text on the far right
for i, row in luck_df.iterrows():
    ax.text(
        x_lim - 0.02,
        i,
        f"{int(row['Wins'])} W | Exp: {row['Expected_Wins']:.2f}",
        va="center",
        ha="right",
        color=MUTED,
        fontsize=9
    )

# Titles
ax.set_title("FFBFFL 2026 — Schedule Luck", color=TEXT, pad=16)
ax.text(
    0.5, 1.02,
    "Through Week 4 | Positive values indicate more wins than expected from all-play performance",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    color=MUTED,
    fontsize=10
)

# Axis labels
ax.set_xlabel("Actual Wins Minus Expected Wins", color=TEXT, labelpad=10)
ax.set_ylabel("")

# Grid / spines
ax.grid(True, axis="x", linestyle=":", linewidth=0.8, alpha=0.45, color=GRID)
for spine in ax.spines.values():
    spine.set_color("#5a6270")
    spine.set_linewidth(1)

ax.tick_params(colors=TEXT)

# Side labels
ax.text(-x_lim, len(luck_df) + 0.35, "More Unlucky", color=RED, fontsize=10, ha="left")
ax.text(x_lim, len(luck_df) + 0.35, "More Lucky", color=GREEN, fontsize=10, ha="right")

plt.tight_layout()

luck_path = os.path.join(OUTPUT_DIR, "ffbffl_2026_schedule_luck.png")
plt.savefig(luck_path, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print("Saved:", luck_path)